In [1]:
import sys
sys.path.insert(1, '/home/bricej/MyPythonLibrary/StageM2_IVT/library/')
import domain
from domain import *
from climbas import *
import numpy as np
import geopandas as gpd
import xarray as xr
import pandas as pd
from netCDF4 import Dataset
import dask
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib as mpl
from shapely.geometry import Polygon, MultiPolygon

# Period

In [2]:
iyear = 1960
fyear = 2019

# Data for the tab

In [3]:
#dhdt 
dhdt = pd.read_csv('/bettik/PROJECTS/pr-regional-climate/bricej/paper/dhdt_2000-2019_HIMAPgrouped_regions.csv')

#era5 variables
data_era5 = {}
variables = ['t2m','FLH','tcwv','ivt','vimd','tp','sf','rf', 'R']

for var in variables:
    data_var = pd.read_csv(f'/bettik/PROJECTS/pr-regional-climate/bricej/paper/{var}_era5_trends_HIMAPgrouped_regions_{iyear}-{fyear}.csv')
    data_var = data_var[["region", "slope", "pvalue"]]
    data_var["slope"] = data_var["slope"] * 10
    data_era5[var] = data_var

# Making the tab with all data to have the overview

In [4]:
# fonction pour arrondir et ajouter * si pvalue < 0.05
def format_trend(value, pvalue, ndigits=2):
    if pd.isna(value):
        return ""
    txt = f"{value:.{ndigits}f}"
    if pvalue < 0.05:
        txt += "*"
    return txt



# Formater dhdt pour avoir value +/- err
df_dh = dhdt.copy()
df_dh["dhdt_str"] = df_dh.apply(
    lambda x: f"{x['dhdt']:.3f} ± {x['err_dhdt']:.3f}",
    axis=1
)
df_final = df_dh[["region", "dhdt_str"]]



# On ajoute les variables era5 pour avoir le tableau final
# nombre de décimales par variable
decimals = {
    "t2m": 2,
    "tp": 1,
    "sf": 1,
    "rf": 1,
    "tcwv": 2,
    "ivt": 2,
    "vimd": 3,
    "FLH": 0,
    "R": 2
}

for var, df_var in data_era5.items():
    nd = decimals.get(var, 2)  # défaut = 2 décimales
    df_tmp = df_var.copy()
    
    df_tmp[var] = df_tmp.apply(
        lambda x: format_trend(x["slope"], x["pvalue"], nd),
        axis=1
    )
    
    df_tmp = df_tmp[["region", var]]
    
    # merge avec le tableau final
    df_final = df_final.merge(df_tmp, on="region", how="left")

In [5]:
df_final

,region,dhdt_str,t2m,FLH,tcwv,ivt,vimd,tp,sf,rf,R
0,Tien Shan,-0.459 ± 0.030,0.20*,24*,0.07*,-0.26,-0.018,0.5,-0.2,0.7*,-0.58*
1,Pamir Alay,-0.122 ± 0.038,0.22*,22*,0.08*,0.13,-0.025,1.3,-0.3,1.6*,-1.12*
2,Pamir,-0.138 ± 0.033,0.23*,23*,0.03*,0.07,-0.010,0.5,-0.3,0.8*,-1.20*
3,Hindu Kush,-0.193 ± 0.035,0.31*,18*,0.05*,-0.01,0.025,-1.3*,-1.3*,0.0,-0.79*
4,Karakoram,-0.073 ± 0.030,0.28*,18*,0.06*,0.19,0.008,0.3,-0.3,0.5*,-0.97*
5,Kunlun,0.080 ± 0.027,0.21*,17*,0.07*,0.13,-0.028*,1.2*,0.1,1.0*,-1.46*
6,Spiti Lahaul,-0.383 ± 0.043,0.29*,18*,0.08*,0.16,0.046,-0.9,-0.7,-0.2,-0.42
7,Central Himalaya,-0.477 ± 0.040,0.23*,23*,0.13*,0.47*,0.040,-0.7,-0.3,-0.4,-0.14
8,Bhutan,-0.538 ± 0.051,0.26*,24*,0.09*,0.41,0.063,-1.9,-0.5*,-1.5,-0.10
9,Nyainqentangla,-0.704 ± 0.062,0.24*,23*,0.06*,0.33,0.051,-1.7,-0.8*,-0.9,-0.22


# Percentage for tcwv, ivt, tp, sf, rf

In [6]:
#era5 variables
data_era5_percentage = {}
variables = ['t2m','FLH','tcwv','ivt','vimd','tp','sf','rf', 'R']

for var in variables:
    if var in ['t2m','FLH','vimd', 'R']:
        data_var = pd.read_csv(f'/bettik/PROJECTS/pr-regional-climate/bricej/paper/{var}_era5_trends_HIMAPgrouped_regions_{iyear}-{fyear}.csv')
        data_var = data_var[["region", "slope", "pvalue"]]
        data_var["slope"] = data_var["slope"] * 10
        data_era5_percentage[var] = data_var

    elif var in ['tcwv','ivt','tp','sf','rf']:
        data_var = pd.read_csv(f'/bettik/PROJECTS/pr-regional-climate/bricej/paper/{var}_era5_percentage_trends_HIMAPgrouped_regions_{iyear}-{fyear}.csv')
        data_var = data_var[["region", "slope", "pvalue"]]
        data_var["slope"] = data_var["slope"] * 10
        data_era5_percentage[var] = data_var

In [7]:
# fonction pour arrondir et ajouter * si pvalue < 0.05
def format_trend(value, pvalue, ndigits=2):
    if pd.isna(value):
        return ""
    txt = f"{value:.{ndigits}f}"
    if pvalue < 0.05:
        txt += "*"
    return txt



# Formater dhdt pour avoir value +/- err
df_dh = dhdt.copy()
df_dh["dhdt_str"] = df_dh.apply(
    lambda x: f"{x['dhdt']:.3f} ± {x['err_dhdt']:.3f}",
    axis=1
)
df_final = df_dh[["region", "dhdt_str"]]



# On ajoute les variables era5 pour avoir le tableau final
# nombre de décimales par variable
decimals = {
    "t2m": 2,
    "tp": 2,
    "sf": 2,
    "rf": 2,
    "tcwv": 2,
    "ivt": 2,
    "vimd": 3,
    "FLH": 0,
    "R": 2
}

for var, df_var in data_era5_percentage.items():
    nd = decimals.get(var, 2)  # défaut = 2 décimales
    df_tmp = df_var.copy()
    
    df_tmp[var] = df_tmp.apply(
        lambda x: format_trend(x["slope"], x["pvalue"], nd),
        axis=1
    )
    
    df_tmp = df_tmp[["region", var]]
    
    # merge avec le tableau final
    df_final = df_final.merge(df_tmp, on="region", how="left")

In [8]:
df_final

,region,dhdt_str,t2m,FLH,tcwv,ivt,vimd,tp,sf,rf,R
0,Tien Shan,-0.459 ± 0.030,0.20*,24*,0.97*,-0.68,-0.018,0.80,-0.79,1.71*,-0.58*
1,Pamir Alay,-0.122 ± 0.038,0.22*,22*,1.05*,0.41,-0.025,1.37,-0.73,3.60*,-1.12*
2,Pamir,-0.138 ± 0.033,0.23*,23*,0.78*,0.30,-0.010,0.92,-0.68,5.10*,-1.20*
3,Hindu Kush,-0.193 ± 0.035,0.31*,18*,0.72*,-0.05,0.025,-1.73*,-3.33*,0.14,-0.79*
4,Karakoram,-0.073 ± 0.030,0.28*,18*,1.81*,1.33,0.008,0.50,-0.66,6.54*,-0.97*
5,Kunlun,0.080 ± 0.027,0.21*,17*,1.66*,0.82,-0.028*,3.21*,0.55,6.17*,-1.46*
6,Spiti Lahaul,-0.383 ± 0.043,0.29*,18*,0.96*,0.67,0.046,-0.98,-1.87,-0.36,-0.42
7,Central Himalaya,-0.477 ± 0.040,0.23*,23*,0.94*,1.30*,0.040,-0.53,-1.54,-0.37,-0.14
8,Bhutan,-0.538 ± 0.051,0.26*,24*,0.63*,0.95,0.063,-0.81,-1.69*,-0.69,-0.10
9,Nyainqentangla,-0.704 ± 0.062,0.24*,23*,0.73*,0.99,0.051,-1.14,-1.88*,-0.83,-0.22


# Sub figures per region

## Inner TP

In [40]:
region = 'Inner TP'

In [41]:
# era5 variables trends
slope1 = []
p1 = []

for var in variables : 
    trend = data_era5[var][data_era5[var]['region'] == region][["region", "slope", "pvalue"]]
    slope1.append({
        "region": region,
        "var": var,
        "slope1": trend["slope"].values[0]
    })

    p1.append({
        "region": region,
        "var": var,
        "p1": trend["pvalue"].values[0]
    })

slope1 = pd.DataFrame(slope1)
p1 = pd.DataFrame(p1)

slope1 = slope1.pivot(
    index="region",
    columns="var",
    values="slope1"
).reset_index()
slope1.columns.name = None

p1 = p1.pivot(
    index="region",
    columns="var",
    values="p1"
).reset_index()
p1.columns.name = None

In [47]:
slope1[['FLH', 't2m', 'tp', 'sf', 'rf', 'R']]

,FLH,t2m,tp,sf,rf,R
0,37.246922,0.3871,-0.055807,-0.863064,0.807257,-1.47829
